<a href="https://colab.research.google.com/github/Priya-Kumari-Chourasia/deep_learning/blob/main/encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.11.0+cpu
Device: cpu


In [44]:
data = [
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is ram", "मेरा नाम राम है"),
    ("where are you going", "आप कहाँ जा रहे हैं"),
    ("i love india", "मुझे भारत से प्यार है"),
    ("i am happy", "मैं खुश हूँ"),
    ("i am sad", "मैं दुखी हूँ"),
    ("he is happy", "वह खुश है"),
    ("she is happy", "वह खुश है"),
    ("this is my house", "यह मेरा घर है"),
    ("this is my book", "यह मेरी किताब है"),
    ("i like tea", "मुझे चाय पसंद है"),
    ("i like coffee", "मुझे कॉफी पसंद है"),
    ("good morning", "सुप्रभात"),
    ("good night", "शुभ रात्रि"),
    ("thank you", "धन्यवाद"),
    ("please help me", "कृपया मेरी मदद करें"),
    ("what are you doing", "आप क्या कर रहे हैं"),
    ("i am learning", "मैं सीख रहा हूँ")
]

print("Number of sentence pairs:", len(data))

Number of sentence pairs: 20


In [45]:
english_sentences = [x[0] for x in data]
hindi_sentences = [x[1] for x in data]

print("English:", english_sentences[0])
print("Hindi  :", hindi_sentences[0])

English: how are you
Hindi  : आप कैसे हैं


In [46]:
def build_vocab(sentences):

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<eos>": 2,
        "<unk>": 3
    }

    for sentence in sentences:

        words = sentence.lower().split()

        for word in words:

            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab

In [47]:
src_vocab = build_vocab(english_sentences)
trg_vocab = build_vocab(hindi_sentences)

print("English vocabulary size:", len(src_vocab))
print("Hindi vocabulary size:", len(trg_vocab))

English vocabulary size: 39
Hindi vocabulary size: 43


In [48]:
print("ENGLISH VOCABULARY")

for word, number in src_vocab.items():
    print(number, "->", word)

ENGLISH VOCABULARY
0 -> <pad>
1 -> <sos>
2 -> <eos>
3 -> <unk>
4 -> how
5 -> are
6 -> you
7 -> i
8 -> am
9 -> fine
10 -> what
11 -> is
12 -> your
13 -> name
14 -> my
15 -> ram
16 -> where
17 -> going
18 -> love
19 -> india
20 -> happy
21 -> sad
22 -> he
23 -> she
24 -> this
25 -> house
26 -> book
27 -> like
28 -> tea
29 -> coffee
30 -> good
31 -> morning
32 -> night
33 -> thank
34 -> please
35 -> help
36 -> me
37 -> doing
38 -> learning


In [49]:
src_itos = {number: word for word, number in src_vocab.items()}
trg_itos = {number: word for word, number in trg_vocab.items()}

In [50]:
print(trg_itos)

{0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>', 4: 'आप', 5: 'कैसे', 6: 'हैं', 7: 'मैं', 8: 'ठीक', 9: 'हूँ', 10: 'आपका', 11: 'नाम', 12: 'क्या', 13: 'है', 14: 'मेरा', 15: 'राम', 16: 'कहाँ', 17: 'जा', 18: 'रहे', 19: 'मुझे', 20: 'भारत', 21: 'से', 22: 'प्यार', 23: 'खुश', 24: 'दुखी', 25: 'वह', 26: 'यह', 27: 'घर', 28: 'मेरी', 29: 'किताब', 30: 'चाय', 31: 'पसंद', 32: 'कॉफी', 33: 'सुप्रभात', 34: 'शुभ', 35: 'रात्रि', 36: 'धन्यवाद', 37: 'कृपया', 38: 'मदद', 39: 'करें', 40: 'कर', 41: 'सीख', 42: 'रहा'}


In [51]:
def numericalize(sentence, vocab):

    words = sentence.lower().split()

    ids = [vocab["<sos>"]]

    for word in words:

        if word in vocab:
            ids.append(vocab[word])
        else:
            ids.append(vocab["<unk>"])

    ids.append(vocab["<eos>"])

    return ids

In [52]:
english = "how are you"

print(numericalize(english, src_vocab))

[1, 4, 5, 6, 2]


In [98]:
def create_tensor(sentences, vocab):

    tensor_list = []

    for sentence in sentences:

        ids = numericalize(sentence, vocab)

        tensor = torch.tensor(
            ids,
            dtype=torch.long
        )

        tensor_list.append(tensor)

    return tensor_list

In [54]:
src_data = create_tensor(
    english_sentences,
    src_vocab
)

trg_data = create_tensor(
    hindi_sentences,
    trg_vocab
)

In [55]:
src_tensor = pad_sequence(
    src_data,
    padding_value=src_vocab["<pad>"]
)

trg_tensor = pad_sequence(
    trg_data,
    padding_value=trg_vocab["<pad>"]
)

In [56]:
print("Source shape:", src_tensor.shape)
print("Target shape:", trg_tensor.shape)

Source shape: torch.Size([6, 20])
Target shape: torch.Size([7, 20])


In [99]:
src = src_tensor.to(device)
trg = trg_tensor.to(device)

print("SRC:", src.shape)
print("TRG:", trg.shape)

SRC: torch.Size([6, 20])
TRG: torch.Size([7, 20])


In [100]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim
        )

        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim
        )


    def forward(self, src):

        embedded = self.embedding(src)

        outputs, (hidden, cell) = self.rnn(
            embedded
        )

        return outputs, hidden, cell

In [101]:
INPUT_DIM = len(src_vocab)

EMBEDDING_DIM = 64

HIDDEN_DIM = 128

In [102]:
encoder = Encoder(
    INPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)

In [103]:
print(encoder)

Encoder(
  (embedding): Embedding(39, 64)
  (rnn): LSTM(64, 128)
)


In [104]:
encoder_outputs, hidden, cell = encoder(src)

print("Encoder outputs:", encoder_outputs.shape)
print("Hidden:", hidden.shape)
print("Cell:", cell.shape)

Encoder outputs: torch.Size([6, 20, 128])
Hidden: torch.Size([1, 20, 128])
Cell: torch.Size([1, 20, 128])


In [105]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.output_dim = output_dim

        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim
        )

        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )


    def forward(
        self,
        input_token,
        hidden,
        cell
    ):

        # input_token:
        # [batch_size]

        input_token = input_token.unsqueeze(0)

        # [1, batch_size]

        embedded = self.embedding(input_token)

        # [1, batch_size, embedding_dim]

        output, (hidden, cell) = self.rnn(
            embedded,
            (hidden, cell)
        )

        # output:
        # [1, batch_size, hidden_dim]

        prediction = self.fc(
            output.squeeze(0)
        )

        # prediction:
        # [batch_size, output_dim]

        return prediction, hidden, cell

In [106]:
OUTPUT_DIM = len(trg_vocab)

DEC_EMBEDDING_DIM = 64

DEC_HIDDEN_DIM = 128

In [107]:
decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMBEDDING_DIM,
    DEC_HIDDEN_DIM
).to(device)

In [108]:
print(decoder)

Decoder(
  (embedding): Embedding(43, 64)
  (rnn): LSTM(64, 128)
  (fc): Linear(in_features=128, out_features=43, bias=True)
)


In [109]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        # Number of sentences
        batch_size = src.shape[1]

        # Hindi sequence length
        trg_len = trg.shape[0]

        # Hindi vocabulary size
        trg_vocab_size = self.decoder.output_dim


        # Store decoder predictions
        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size
        ).to(self.device)


        # -----------------------
        # ENCODER
        # -----------------------

        encoder_outputs, hidden, cell = self.encoder(src)


        # First decoder input
        # = <sos>

        input_token = trg[0, :]


        # -----------------------
        # DECODER
        # -----------------------

        for t in range(1, trg_len):

            output, hidden, cell = self.decoder(
                input_token,
                hidden,
                cell
            )

            outputs[t] = output


            # Model's predicted word

            best_prediction = output.argmax(1)


            # Teacher forcing

            if random.random() < teacher_forcing_ratio:

                input_token = trg[t, :]

            else:

                input_token = best_prediction


        return outputs

In [110]:
encoder = Encoder(
    INPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)


decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMBEDDING_DIM,
    DEC_HIDDEN_DIM
).to(device)


model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

In [111]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=trg_vocab["<pad>"]
)

In [113]:
import random
output = model(
    src,
    trg
)

print("SRC:", src.shape)
print("TRG:", trg.shape)
print("OUTPUT:", output.shape)

SRC: torch.Size([6, 20])
TRG: torch.Size([7, 20])
OUTPUT: torch.Size([7, 20, 43])


In [114]:
output.argmax(1)

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0],
        [ 7, 11,  4, 16,  2, 16, 11, 15, 11, 10,  3,  3, 10, 10,  4, 15, 14, 10,
         18,  4, 12,  4, 16,  4, 18,  2, 19, 17,  2, 18, 10,  3,  3, 11, 11,  3,
         11, 14, 10, 14,  3, 11, 18],
        [15,  2,  4, 14,  2,  0,  4, 13, 11, 18,  0, 14,  4, 10, 16, 16, 14,  3,
         10, 14,  3, 15, 14,  9, 11, 15,  5,  2, 17,  2,  2,  3,  1,  3,  6, 10,
          3, 17,  3,  3,  1,  2,  0],
        [ 7, 18, 13, 14, 12,  0, 19,  1,  1, 15,  7,  0,  0,  7, 16, 19, 17, 10,
         10,  1,  3, 15, 14,  4,  4, 15, 17, 15,  0,  2, 18,  1,  3, 10,  6, 11,
         10, 19, 16,  5,  5, 15, 18],
        [ 7,  7,  1,  5,  7, 15,  0,  1, 15,  7,  6, 15,  0,  7, 16,  8,  3,  0,
          1,  1,  3,  3,  7, 17,  5,  1,  1,  7, 13,  2,  2, 10,  5,  6, 11, 11,
         11,  9, 17,  5,  5, 12,  1],


In [115]:
def train(
    model,
    src,
    trg,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()


        # Forward pass

        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )


        # Remove first token (<sos>)

        output_dim = output.shape[-1]

        output = output[1:].reshape(
            -1,
            output_dim
        )


        # Target also removes <sos>

        trg_y = trg[1:].reshape(-1)


        # Calculate loss

        loss = criterion(
            output,
            trg_y
        )


        # Backpropagation

        loss.backward()


        # Update weights

        optimizer.step()


        if (epoch + 1) % 50 == 0:

            print(
                f"Epoch: {epoch+1:3d} | "
                f"Loss: {loss.item():.4f}"
            )

In [116]:
train(
    model,
    src,
    trg,
    optimizer,
    criterion,
    epochs=500
)

Epoch:  50 | Loss: 1.2046
Epoch: 100 | Loss: 0.1487
Epoch: 150 | Loss: 0.0434
Epoch: 200 | Loss: 0.0226
Epoch: 250 | Loss: 0.0144
Epoch: 300 | Loss: 0.0102
Epoch: 350 | Loss: 0.0077
Epoch: 400 | Loss: 0.0060
Epoch: 450 | Loss: 0.0049
Epoch: 500 | Loss: 0.0040


In [117]:
def translate_sentence(
    sentence,
    model,
    src_vocab,
    trg_vocab,
    trg_itos,
    device,
    max_len=20
):

    model.eval()


    # Convert English sentence to IDs

    src_ids = numericalize(
        sentence,
        src_vocab
    )


    # Tensor shape:
    # [sequence_length]

    src_tensor = torch.tensor(
        src_ids,
        dtype=torch.long
    ).unsqueeze(1).to(device)


    # -----------------------
    # ENCODER
    # -----------------------

    with torch.no_grad():

        encoder_outputs, hidden, cell = model.encoder(
            src_tensor
        )


    # First decoder input = <sos>

    input_token = torch.tensor(
        [trg_vocab["<sos>"]],
        dtype=torch.long
    ).to(device)


    translated_words = []


    # -----------------------
    # DECODER
    # -----------------------

    for _ in range(max_len):

        with torch.no_grad():

            output, hidden, cell = model.decoder(
                input_token,
                hidden,
                cell
            )


        # Select most likely word

        predicted_token = output.argmax(1).item()


        # Stop if <eos>

        if predicted_token == trg_vocab["<eos>"]:
            break


        # Convert ID → Hindi word

        predicted_word = trg_itos[predicted_token]

        translated_words.append(
            predicted_word
        )


        # Feed predicted word
        # back into decoder

        input_token = torch.tensor(
            [predicted_token],
            dtype=torch.long
        ).to(device)


    return " ".join(translated_words)

In [118]:
result = translate_sentence(
    "how are you",
    model,
    src_vocab,
    trg_vocab,
    trg_itos,
    device
)

print("English:", "how are you")
print("Hindi:", result)

English: how are you
Hindi: आप कैसे हैं
